In [1]:
import pandas as pd
import numpy as np
import psycopg
from dotenv import load_dotenv
import os
import time

## 1) Pobranie danych i zapisanie do bazy danych

In [262]:
load_dotenv()
def make_connection():
    user = os.getenv("DBUSER")
    passwd = os.getenv("DBPASSWORD")
    host = os.getenv("DBHOST")
    dbname = os.getenv("DBNAME")

    try:
        return psycopg.connect(f"dbname={dbname} user={user} host={host} password={passwd}")
    except:
        return False

In [263]:
conn = make_connection()

if conn:
    print("Połączono!")
else:
    print("Nie udało się!")

Połączono!


In [268]:
df.interpolate().bfill().to_sql("rates", conn, "rates", "append")

C:\Users\AdamD\AppData\Local\Temp\ipykernel_11544\4147585422.py:1: FutureWarning: Starting with pandas version 3.0 all arguments of to_sql except for the arguments 'name' and 'con' will be keyword-only.
  df.interpolate().bfill().to_sql("rates", conn, "rates", "append")
C:\Users\AdamD\AppData\Local\Temp\ipykernel_11544\4147585422.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df.interpolate().bfill().to_sql("rates", conn, "rates", "append")


DatabaseError: Execution failed on sql '
        SELECT
            name
        FROM
            sqlite_master
        WHERE
            type IN ('table', 'view')
            AND name=?;
        ': the query has 0 placeholders but 1 parameters were passed

In [269]:
conn.close()

## Pobieranie danych z API

### Wybrane waluty:
- korona czeska (CZK)
- dolar amerykański (USD)
- euro (EUR)
- forint (HUF)
- jen (JPY)
- hrywna (UAH)

In [2]:
import requests
import json
from io import StringIO

In [3]:
resp = requests.get(r"https://api.nbp.pl/api/exchangerates/tables/A/2026-04-09/2026-06-11", headers = {'Accept': 'application/json'})
try:
    assert resp.status_code == 200
except:
    print(resp.status_code)


In [4]:
codes = ['USD', 'EUR', 'HUF', 'JPY', 'UAH', 'CZK']
data = resp.json()

In [5]:
def filter_rates(data):
    filtered = []
    for row in data:
        
        row_dict = {}
        row_dict['date'] = row['effectiveDate']
        rates = row['rates']
        filtered_rates = [*filter(lambda x: x['code'] in codes, rates)]
    
        for rate in filtered_rates:
            row_dict[f"{rate['code']}"] = rate['mid']
        
        filtered.append(row_dict)

    return filtered

In [6]:
filtered = filter_rates(data)

In [7]:
data_buff = StringIO(json.dumps(filtered))

In [8]:
df = pd.read_json(data_buff, orient = "records")
df['date'] = pd.to_datetime(df['date'])

In [9]:
df.set_index("date", inplace = True)

In [10]:
df

,USD,EUR,HUF,UAH,JPY,CZK
date,,,,,,
2026-04-09,3.6506,4.2610,0.011269,0.0840,0.022959,0.1746
2026-04-10,3.6396,4.2534,0.011287,0.0838,0.022845,0.1745
2026-04-13,3.6374,4.2507,0.011602,0.0837,0.022775,0.1744
2026-04-14,3.6015,4.2436,0.011681,0.0828,0.022642,0.1742
2026-04-15,3.5939,4.2370,0.011648,0.0826,0.022635,0.1740
2026-04-16,3.5983,4.2411,0.011629,0.0825,0.022633,0.1742
2026-04-17,3.5912,4.2355,0.011639,0.0818,0.022566,0.1742
2026-04-20,3.6005,4.2346,0.011678,0.0816,0.022647,0.1743
2026-04-21,3.5970,4.2320,0.011698,0.0815,0.022598,0.1742


In [11]:
from sklearn.preprocessing import StandardScaler

sc = StandardScaler()
df_scaled = pd.DataFrame(sc.fit_transform(df), index = df.index, columns = df.columns)

In [12]:
from datetime import date, timedelta, datetime

datetime.strftime

<method 'strftime' of 'datetime.date' objects>

In [13]:
def make_date_chunks(start_date: date, end_date: date):
    """
    podział zakresu dat na okresy maksymalnie 90-dniowe (związane z limitami pobierania danych z API)
    """
    date_format = "%Y-%m-%d"
    
    dt = timedelta(days = 89)
    no_of_days = (end_date - start_date).days

    # ilość okresów o największej długości
    no_of_max_chunks = no_of_days // 90

    # ilość dat w ostatnim okresie
    last_chunk_size = no_of_days % 90

    # wynikiem będzie lista krotek z datą pierwszą i ostatnią danego okresu
    period_list = []

    # tworzenie zakresów
    curr_date = start_date
    for n in range(no_of_max_chunks):
        
        curr_date_str = curr_date.strftime(date_format)
        next_date_str = (curr_date + dt).strftime(date_format)
        period_tuple = (curr_date_str, next_date_str)
        
        period_list.append(period_tuple)
        curr_date = curr_date + timedelta(days = 90)

    # stworzenie ostatniego zakresu (o długości < od maksymalnej długości)
    if last_chunk_size:
        curr_date_str = curr_date.strftime(date_format)
        next_date_str = (curr_date + timedelta(days = last_chunk_size)).strftime(date_format)
        period_tuple = (curr_date_str, next_date_str)
        period_list.append(period_tuple)

    return period_list

In [14]:
make_date_chunks(d1,d2)

NameError: name 'd1' is not defined

In [ ]:
def download_data(start_date_str, end_date_str):

    date_format = "%Y-%m-%d"

    start_date = datetime.strptime(start_date_str, date_format)
    end_date = datetime.strptime(end_date_str, date_format)

    periods = make_date_chunks(start_date, end_date)

    dates = pd.date_range(start_date, end_date)

    df = pd.DataFrame(columns = codes, index = dates)
    
    for period in periods:
        
        d1, d2 = period
        
        url = r"https://api.nbp.pl/api/exchangerates/tables/A/{}/{}".format(d1, d2)
        
        resp = requests.get(url, headers = {'Accept': 'application/json'})
        try:
            assert resp.status_code == 200
        except:
            print("Błąd pobierania!")
            print(resp.status_code)
            return

        data = resp.json()

        data = filter_rates(data)

        data_str = json.dumps(data)
        buff = StringIO(data_str)

        temp = pd.read_json(buff, orient = "records")
        temp.set_index("date", inplace = True)
        
        ix = temp.index.strftime(date_format)
        
        try:
            df.loc[ix] = temp.values
        except:
            pass
            
        print("poszło!")
        time.sleep(3)
    
    return df.astype("float")
        

        

In [15]:
def clean_data(data):

    return data.interpolate().bfill()

In [259]:
pd.DataFrame([np.nan, 1,2,3,4, np.nan]).interpolate()

,0
0,NaN
1,1.0
2,2.0
3,3.0
4,4.0
5,4.0


In [21]:
import pandas as pd
from datetime import date, timedelta, datetime
import requests
import json 
from io import StringIO
import time

# kody walut, które będą analizowane
codes = ['usd', 'eur', 'huf', 'jpy', 'uah', 'czk']


def make_date_chunks(start_date: date, end_date: date):
    """
    podział zakresu dat na okresy maksymalnie 90-dniowe (związane z limitami pobierania danych z API)
    """
    date_format = "%Y-%m-%d"
    
    dt = timedelta(days = 89)
    no_of_days = (end_date - start_date).days

    # ilość okresów o największej długości
    no_of_max_chunks = no_of_days // 90

    # ilość dat w ostatnim okresie
    last_chunk_size = no_of_days % 90

    # wynikiem będzie lista krotek z datą pierwszą i ostatnią danego okresu
    period_list = []

    # tworzenie zakresów
    curr_date = start_date
    for n in range(no_of_max_chunks):
        
        curr_date_str = curr_date.strftime(date_format)
        next_date_str = (curr_date + dt).strftime(date_format)
        period_tuple = (curr_date_str, next_date_str)
        
        period_list.append(period_tuple)
        curr_date = curr_date + timedelta(days = 90)

    # stworzenie ostatniego zakresu (o długości < od maksymalnej długości)
    if last_chunk_size:
        curr_date_str = curr_date.strftime(date_format)
        next_date_str = (curr_date + timedelta(days = last_chunk_size)).strftime(date_format)
        period_tuple = (curr_date_str, next_date_str)
        period_list.append(period_tuple)

    return period_list



def filter_rates(data, codes = codes):
    """
    Wyciąga z pliku json wyłącznie potrzebne informacje (data i kursy wybranyc walut)
    """
    filtered = []
    for row in data:
        
        row_dict = {}
        row_dict['date'] = row['effectiveDate']
        rates = row['rates']
        filtered_rates = [*filter(lambda x: x['code'].lower() in codes, rates)]
    
        for rate in filtered_rates:
            row_dict[f"{rate['code'].lower()}"] = rate['mid']
        
        filtered.append(row_dict)

    return filtered


def download_data(start_date_str, end_date_str):

    """
    Funkcja do pobierania danych - zwraca dataframe
    """
    
    
    date_format = "%Y-%m-%d"

    start_date = datetime.strptime(start_date_str, date_format)
    end_date = datetime.strptime(end_date_str, date_format)

    periods = make_date_chunks(start_date, end_date)

    dates = pd.date_range(start_date, end_date)

    df = pd.DataFrame(columns = codes, index = dates)
    
    for period in periods:
        
        d1, d2 = period
        
        url = r"https://api.nbp.pl/api/exchangerates/tables/A/{}/{}".format(d1, d2)
        
        resp = requests.get(url, headers = {'Accept': 'application/json'})
        try:
            assert resp.status_code == 200
        except:
            print("Błąd pobierania!")
            print(resp.status_code)
            return

        data = resp.json()

        data = filter_rates(data)

        data_str = json.dumps(data)
        buff = StringIO(data_str)

        temp = pd.read_json(buff, orient = "records")
        temp.set_index("date", inplace = True)
        
        ix = temp.index.strftime(date_format)
        
        try:
            df.loc[ix] = temp.values
        except:
            pass
            
        print("poszło!")
        time.sleep(3)
    
    
    return df.astype("float")


def clean_data(data):

    """
    Uzupełnianie braków danych - interpolacja (oraz uzupełnienie wartością następną dla braków na samym początku)
    """

    return data.interpolate().bfill()


def get_data(start_date_str, end_date_str):

    df_raw = download_data(start_date_str, end_date_str)

    df = clean_data(df_raw)
    
    return df

In [22]:
get_data("2022-02-03", "2022-05-06")

poszło!
poszło!


,usd,eur,huf,jpy,uah,czk
2022-02-03,NaN,NaN,NaN,NaN,NaN,NaN
2022-02-04,NaN,NaN,NaN,NaN,NaN,NaN
2022-02-05,NaN,NaN,NaN,NaN,NaN,NaN
2022-02-06,NaN,NaN,NaN,NaN,NaN,NaN
2022-02-07,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
2022-05-02,NaN,NaN,NaN,NaN,NaN,NaN
2022-05-03,NaN,NaN,NaN,NaN,NaN,NaN
2022-05-04,NaN,NaN,NaN,NaN,NaN,NaN
2022-05-05,NaN,NaN,NaN,NaN,NaN,NaN
